# Livskompassen v2 - Admin & Datasortering
Denna notebook används som din isolerade, AI-drivna arbetsyta. Här kan du:
* Koppla upp dig mot Google Drive för att hantera projektfiler och dokument.
* Autentisera säkert mot ditt Google Cloud-projekt.
* Förbereda, tvätta och skicka data direkt till ditt Vertex AI 'Kunskapsvalv'.

In [ ]:
from google.colab import drive
from google.colab import auth
import os

# 1. Montera Google Drive. Du kommer att få en popup som ber om behörighet.
drive.mount('/content/drive')

# 2. Autentisera din användare. Detta säkerställer att Colab får använda dina 12 000 kr krediter.
auth.authenticate_user()
print("\n✅ Autentisering och Drive-montering lyckades!")

In [ ]:
import vertexai

# --- VIKTIGT: BYT UT DETTA MOT DITT FAKTISKA PROJEKT-ID ---
PROJECT_ID = "ditt-google-cloud-project-id"
LOCATION = "europe-west1"

# 3. Initiera Vertex AI
vertexai.init(project=PROJECT_ID, location=LOCATION)
print(f"✅ Vertex AI anslutet till projekt: {PROJECT_ID}")

In [ ]:
# 4. Installera bibliotek för att läsa PDF-filer
# '-q' flaggan gör installationen tyst (mindre output)
!pip install pypdf -q
print("✅ Biblioteket 'pypdf' är installerat.")

In [ ]:
import pypdf
import re
from vertexai.language_models import TextEmbeddingModel

# 5. Definiera hjälpfunktioner för text-extrahering och embeddings

# Ladda embedding-modellen en gång för att återanvända den (mer effektivt)
embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-004")

def extract_text_from_pdf(pdf_path: str) -> str:
    """Öppnar en PDF-fil och extraherar all text."""
    print(f"  -> Extraherar text från: {os.path.basename(pdf_path)}")
    try:
        reader = pypdf.PdfReader(pdf_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""
        
        # Enkel rensning: ersätt flera blanksteg/radbrytningar med ett enda mellanslag
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    except Exception as e:
        print(f"    -> Kunde inte läsa PDF: {e}")
        return ""

def get_embedding(text: str) -> list[float] | None:
    """Skapar en embedding (vektor) från en textsträng."""
    if not text:
        return None
    try:
        embeddings = embedding_model.get_embeddings([text])
        return embeddings[0].values
    except Exception as e:
        print(f"    -> Kunde inte skapa embedding: {e}")
        return None

print("✅ Funktioner för PDF-hantering och embeddings är redo.")

In [ ]:
# 6. Bearbeta PDF-filer från Drive och skapa embeddings
drive_path = '/content/drive/MyDrive/Livskompassen_Data/'
os.makedirs(drive_path, exist_ok=True)

print(f"Söker efter PDF-filer i: {drive_path}...")

processed_documents = []
files_in_drive = os.listdir(drive_path)

if not files_in_drive:
    print("Mappen är tom. Lägg till PDF-filer i din Drive-mapp för att analysera dem.")
else:
    for file_name in files_in_drive:
        if file_name.lower().endswith(".pdf"):
            print(f"Hittade PDF: {file_name}")
            full_path = os.path.join(drive_path, file_name)
            
            document_text = extract_text_from_pdf(full_path)
            
            if document_text:
                document_embedding = get_embedding(document_text)
                
                if document_embedding:
                    # Skapa ett unikt ID från filnamnet (t.ex. 'min_fil.pdf' -> 'min_fil')
                    doc_id = os.path.splitext(file_name)[0].replace(" ", "_").lower()
                    
                    processed_documents.append({
                        "id": doc_id,
                        "text": document_text,
                        "embedding": document_embedding
                    })
                    print(f"  -> ✅ Bearbetning klar för '{doc_id}'")

print(f"\n--- Resultat ---")
print(f"Totalt antal bearbetade dokument: {len(processed_documents)}")
if processed_documents:
    print("Exempel på första dokumentet:")
    # Skriv inte ut hela embeddingen, den är för lång
    print(f"  ID: {processed_documents[0]['id']}")
    print(f"  Text: {processed_documents[0]['text'][:150]}...")
    print(f"  Embedding-dimension: {len(processed_documents[0]['embedding'])}")
    
# Nästa steg: Använd 'processed_documents' för att skapa en JSONL-fil 
# och ladda upp till GCS, precis som i 'setup_knowledge_vault.py'.